# 06 — Kommersiell marin trafikanalys

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TobiasEdman/space-data-lab/main?labpath=notebooks%2Fsdl3-cases%2F070_CASE_070-Commercial-Marine-Oresund.ipynb)

**Syfte:** Demonstrera hur kommersiell VHR-data (Maxar) kombineras med Sentinel-2 och onboard AI (PandionAI) för att skapa detaljerad marin trafikövervakning. Skillnaden mot [02_marin_sotenas](070_CASE_030-Marine-Vessels-Sotenas.ipynb): här ligger fokus på kommersiell datakedja och förfining via VHR.

**Partners som bidragit:**
- **Maxar International Sweden** — VHR (WorldView-2/3, sub-meter upplösning)
- **PandionAI** — AlertSat onboard AI, realtidsdetektion
- **SSC (Swedish Space Corporation)** — markstationer, tasking
- **AI Sweden** — Edge Lab, onboard AI-modellutveckling

**Datakällor:**
- Sentinel-2 L2A (10 m) — initial screening via DES
- Maxar WorldView (~0.3 m pan, ~1.2 m MS) — detaljkarakterisering (kommersiell licens)
- AI2 vessel detection + fine-tuned YOLO11s

**Licens:** CC0 1.0 (notebook) · Maxar-data kräver kommersiell licens

## ⚠️ Licens — YOLO11s är AGPL-3.0

Denna notebook använder `marine_vessels`-analyzern som bygger på **YOLO11s** (Ultralytics, AGPL-3.0).

**Vad det innebär:**

| Användning | OK utan Enterprise License? |
|-----------|------------------------------|
| Binder / publik Jupyter (denna notebook) | ✅ Ja — vårt repo är publikt, AGPL-nätverksklausulen uppfylld |
| Lokal körning för forskning/utbildning | ✅ Ja — ingen distribution |
| DES JupyterHub (publik plattform) | ✅ Ja — open-source infrastruktur |
| Kommersiell sluten produkt eller betaltjänst | ❌ **Nej** — kräver [Ultralytics Enterprise License](https://www.ultralytics.com/license) |

**Alternativ utan AGPL:** `ai2_vessels` (Allen Institute-modell) fungerar som ersättare med annan licensprofil. `object_detection` i heatmap-läge är helt modellfri (ren variansmatematik) och kan användas för first-pass-screening utan licensbörda.

Se [ImintEngine THIRD_PARTY_LICENSES.md](https://github.com/TobiasEdman/imintengine/blob/main/THIRD_PARTY_LICENSES.md) för fullständig licensöversikt.

## 1. Metod — tiered detection

**Tre-stegs pipeline:**

```
Steg 1: Sentinel-2 (10 m, gratis)
        → NDWI-mask, YOLO screening
        → Heatmap över fartygstäthet
        → Identifiera AOI för detaljanalys

Steg 2: Maxar VHR (0.3 m, licensierad)
        → Detaljanalys av hotspots
        → Klass, storlek, orientering
        → Referensklassificering vs AIS

Steg 3: PandionAI onboard (realtid)
        → Förändringsdetektion on-orbit
        → Downlink endast avvikelser → sparar bandbredd
```

Denna *tiered approach* visar hur kommersiella och öppna data samverkar — en direkt koppling till SDL 3.0-ansökans åtagande om VHR-data och onboard AI.

## 2. Setup

In [ ]:
# Verified API helper — wraps IMINTResult access
def get_outputs(result, name):
    """Return outputs dict for first successful analyzer matching `name`, else None."""
    for ar in result.analyzer_results:
        if ar.analyzer == name and ar.success:
            return ar.outputs
    return None


In [ ]:
from executors.local import LocalExecutor
import matplotlib.pyplot as plt
import folium

# Öresund — tät kommersiell sjöfart, bra för showcase
AOI = {
    "west": 12.65,
    "south": 55.60,
    "east": 12.85,
    "north": 55.80,
}
DATE = "2024-07-20"

print(f"AOI (Öresund): {AOI}")
print(f"Datum: {DATE}")

## 3. Steg 1 — Sentinel-2 screening

In [ ]:
executor = LocalExecutor(
    output_dir="outputs/marine_commercial",
    config_path="config/analyzers.yaml",
)

result = executor.execute(
    date=DATE,
    coords=AOI,
)

# Täthetshistogram
if get_outputs(result, "marine_vessels") is not None:
    vessels = get_outputs(result, "marine_vessels").data.get("detections", [])
    print(f"YOLO11s: {len(vessels)} fartyg")
if "ai2_vessels" in result.analyses:
    ai2 = get_outputs(result, "ai2_vessels").data.get("detections", [])
    print(f"AI2: {len(ai2)} fartyg")

## 4. Visualisering — heatmap + detektionskarta

In [ ]:
center = [(AOI["south"] + AOI["north"]) / 2, (AOI["west"] + AOI["east"]) / 2]
m = folium.Map(location=center, zoom_start=11, tiles="OpenStreetMap")
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri", name="Satellit",
).add_to(m)

if get_outputs(result, "marine_vessels") is not None:
    for v in get_outputs(result, "marine_vessels").data.get("detections", []):
        if "lat" in v and "lon" in v:
            folium.CircleMarker(
                location=[v["lat"], v["lon"]],
                radius=3, color="red", fill=True,
                popup=f"YOLO conf: {v.get('confidence', 0):.2f}",
            ).add_to(m)

folium.LayerControl().add_to(m)
m

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(result.rgb)
axes[0].set_title("Sentinel-2 RGB")
axes[0].axis("off")

if get_outputs(result, "marine_vessels") is not None:
    heatmap = get_outputs(result, "marine_vessels").data.get("heatmap")
    if heatmap is not None:
        axes[1].imshow(heatmap, cmap="hot")
        axes[1].set_title("YOLO11s täthets-heatmap")
axes[1].axis("off")

if "ai2_vessels" in result.analyses:
    ai2_heatmap = get_outputs(result, "ai2_vessels").data.get("heatmap")
    if ai2_heatmap is not None:
        axes[2].imshow(ai2_heatmap, cmap="hot")
        axes[2].set_title("AI2 täthets-heatmap")
axes[2].axis("off")

plt.tight_layout()
plt.show()

## 5. Tolkning — kommersiell dataaffär

**Varför tiered detection är nyckeln till SDL 3.0:**

| Nivå | Kostnad | Upplösning | Täckning | Användning |
|------|---------|-----------|----------|-----------|
| Sentinel-2 | 0 SEK | 10 m | Global, 5 dagar | Screening, trend |
| Maxar WV-2/3 | ~40 USD/km² | 0.3–1.2 m | On-demand | Detaljanalys, bevis |
| PandionAI onboard | Prenumeration | Varierar | Realtid | Larm, händelsedetektion |

**Kommersiell kedja (SDL 3.0-ansökan):**
- Maxar: 3–4 masterexamensarbeten (NeRF, GAN, sensor fusion)
- PandionAI: Bidrar tid + thesis-finansiering + value-added data
- SSC: Markstationer + kommersiella tjänster
- Tillsammans: visar hur offentligt finansierad infrastruktur (Copernicus, DES) skalas av kommersiella tjänster

**Nästa steg:**
- Fusion med AIS-data → identifiera "dark vessels"
- Komplettera med Sentinel-1 SAR för molnfri täckning
- Integrera onboard-detektion från PandionAlert
- Tracking över tidsserier → rutanalys

### Länkar
- [Maxar WorldView-2/3](https://www.maxar.com)
- [PandionAI AlertSat](https://www.pandionai.com)
- [SSC Satellite Ground Stations](https://sscspace.com)
- [AI2 Satellite Vessel Detection](https://allenai.org)